# 04 · Circuit validation and resource analysis

**Scientific question.** What does a boundary-aware split step cost as a circuit, under stated and comparable assumptions?

**Scope.** End-to-end small-system circuit validation, structured phase synthesis, resource scaling, connectivity, approximate QFT, and the cost of state preparation and measurement.

**Inputs.** `configs/{PROFILE}.yaml` (loaded below). No other notebook needs to have
been run first: this notebook imports everything it needs from
`src/boundary_aware_dynamics` and holds no state from any other notebook.

**Expected outputs.** Resource tables by component, scaling in register size, and an approximation trade-off study.

**Approximate runtime.** about 60 seconds on the `smoke` profile.

**Method.** Circuits are validated against the numerical propagator at 2–3 data qubits, then counted after transpilation with the seed, basis, optimisation level and connectivity all recorded.

**Assumptions.** Counts are logical, not fault-tolerant. State preparation is exact amplitude encoding and is reported separately. No hardware is involved.

**References.** See `references/references.bib` and `docs/SCIENTIFIC_METHOD.md`.

**What this notebook does _not_ establish.** **No end-to-end speedup claim.** A polynomial-size transform does not by itself make an algorithm efficient; state preparation, potential synthesis, measurement and shot cost all remain, and are reported rather than assumed away.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt

from boundary_aware_dynamics.config import load_config
from boundary_aware_dynamics import plotting

PROFILE = os.environ.get("BAD_PROFILE", "smoke")   # "paper" for manuscript numbers
                                                   # (scripts/execute_notebooks.py sets this)
config = load_config(ROOT / "configs" / f"{PROFILE}.yaml")
plotting.apply_style("preview")
print(f"profile={config.profile}  config_hash={config.config_hash}")

## End-to-end circuit equals the numerical propagator

A full multi-step split circuit — extension, transform, kinetic phase, inverse transform, uncomputation, potential phases — is compared against the numerical propagation of the same state.

In [ ]:
from qiskit.quantum_info import Statevector
from boundary_aware_dynamics.circuits.resources import build_propagation_core
from boundary_aware_dynamics.circuits.phases import (
    folded_sine_kinetic_expansion, linear_tilt_expansion)
from boundary_aware_dynamics.circuits.state_preparation import exact_state_preparation_circuit
from boundary_aware_dynamics.grids import dirichlet_midpoint_grid
from boundary_aware_dynamics.propagators import split_operator_evolution, tilted_potential
from boundary_aware_dynamics.states import sine_windowed_gaussian, physical_to_amplitudes
from qiskit import QuantumCircuit

L, F = 10.0, 5.0
mass, hbar = config.physics.mass, config.physics.hbar
for n, steps in ((2, 1), (2, 3), (3, 2)):
    N = 2**n; dt = 0.05
    g = dirichlet_midpoint_grid(L, N)
    psi0 = sine_windowed_gaussian(g.positions, g.spacing, L, 5.0, 2.0, 1.2)
    V = tilted_potential(g.positions, L, F)

    core = build_propagation_core(
        "dirichlet", N, steps,
        linear_tilt_expansion(N, L, F, hbar, dt),
        folded_sine_kinetic_expansion(N, L, mass, hbar, dt),
        linear_tilt_expansion(N, L, F, hbar, 0.5 * dt))

    full = QuantumCircuit(n + 2)
    full.compose(exact_state_preparation_circuit(psi0, g.spacing), qubits=range(1, n + 1), inplace=True)
    full.compose(core, inplace=True)
    got = Statevector(full).data[2 * np.arange(N)]

    want = physical_to_amplitudes(
        split_operator_evolution(psi0, g, V, steps * dt, steps, mass, hbar).final_state, g.spacing)
    overlap = abs(np.vdot(want, got))
    print(f"n={n} data qubits, r={steps}: |<numerical|circuit>| = {overlap:.12f}   "
          f"ancilla leakage = {np.linalg.norm(np.delete(Statevector(full).data, 2*np.arange(N))):.2e}")

## Structured synthesis versus a generic diagonal

A generic `DiagonalGate` costs $O(2^n)$. Every diagonal needed here is quadratic in the register index and factorises into $O(n^2)$ gates. The generic figure is retained only as an explicitly labelled upper bound.

In [ ]:
from boundary_aware_dynamics.circuits.resources import propagation_resources
import pandas as pd

rows = []
for name in ("harmonic", "tilted_well"):
    for syn in ("structured", "generic_diagonal"):
        r = propagation_resources(config, name, n_steps=1, synthesis=syn)
        rows.append({"benchmark": name, "synthesis": syn, "data": r.n_data_qubits,
                     "ancilla": r.n_ancilla_qubits, "total": r.n_total_qubits,
                     "1q": r.one_qubit_gates, "2q": r.two_qubit_gates, "depth": r.total_depth})
display(pd.DataFrame(rows))

## Step composition

$r$ steps are **not** $r$ copies of the five-block single step: adjacent half-potential phases merge, giving one initial half-phase, $r-1$ full phases, one final half-phase and $r$ kinetic blocks.

In [ ]:
rows = []
for r_ in (1, 2, 4, 8, 16):
    row = propagation_resources(config, "tilted_well", n_steps=r_)
    rows.append({"r": r_, "1q": row.one_qubit_gates, "2q": row.two_qubit_gates,
                 "depth": row.total_depth, "composition": row.notes})
display(pd.DataFrame(rows))

## Scaling and connectivity

One qubit count is not a scaling study, and a gate count without a connectivity assumption is not comparable to anything.

In [ ]:
from boundary_aware_dynamics.circuits.resources import scaling_table

rows = []
for name in ("harmonic", "tilted_well"):
    for conn in ("all_to_all", "linear"):
        for row in scaling_table(config, name, (8, 16, 32, 64), n_steps=1, connectivity=conn):
            rows.append({"benchmark": name, "connectivity": conn, "N": row.n_grid,
                         "total_qubits": row.n_total_qubits, "2q": row.two_qubit_gates,
                         "2q_depth": row.two_qubit_depth})
frame = pd.DataFrame(rows)
display(frame.pivot_table(index=["benchmark", "N"], columns="connectivity", values="2q"))

In [ ]:
sub = frame[(frame.benchmark == "tilted_well") & (frame.connectivity == "all_to_all")]
sub2 = frame[(frame.benchmark == "harmonic") & (frame.connectivity == "all_to_all")]
fig = plotting.plot_resource_scaling(
    sub.total_qubits.values,
    {"Dirichlet (QST, +2 ancillas)": sub["2q"].values,
     "Periodic (QFT, no ancillas)": sub2["2q"].values})
plt.show()

## Approximate QFT: the trade-off, not just the saving

In [ ]:
from boundary_aware_dynamics.circuits.resources import approximate_qft_study
display(pd.DataFrame(approximate_qft_study(config, "harmonic", degrees=(0, 1, 2, 3), n_steps=1)))

## What is *not* in the propagation core

State preparation and measurement are real costs and are reported separately rather than folded in.

In [ ]:
from boundary_aware_dynamics.circuits.resources import count_resources
from boundary_aware_dynamics.circuits.state_preparation import measurement_note, shot_budget_for_density

for n in (3, 4, 5, 6):
    N = 2**n
    g = dirichlet_midpoint_grid(L, N)
    psi = sine_windowed_gaussian(g.positions, g.spacing, L, 5.0, 2.0, 0.8)
    c = count_resources(exact_state_preparation_circuit(psi, g.spacing), config)
    print(f"  exact state preparation, {n} qubits (N={N:3d}): {c['two_qubit_gates']:4d} two-qubit gates")
print("\n-> exponential in the qubit count; no efficient-preparation claim is made anywhere.\n")
for k, v in measurement_note().items():
    print(f"  {k}: {v}")
print(f"\n  shots for 1% density precision at 99% confidence: {shot_budget_for_density(0.01)}")

## Summary

**Main findings.** Full multi-step circuits reproduce the numerical propagator to machine precision with ancillas returned cleanly. Structured synthesis reduces the two-qubit count substantially against a generic diagonal, merged composition avoids double-counting the potential phases, and two-qubit cost grows polynomially in the register size. Linear connectivity costs roughly 2.5× the all-to-all count.

**Validation checks performed.** Circuit-versus-numerical agreement at 2–3 data qubits over several step counts; ancilla leakage; structured-versus-generic comparison; composition accounting; connectivity variants; approximate-QFT error against gate saving.

**Limitations.** Logical counts only — no fault-tolerant overhead, no error correction, no hardware execution. Exact state preparation is exponential and is not claimed to be efficient. Unitary validation is limited to register sizes where the operator can be formed.

**Generated files.** None directly; notebook 05 exports the resource tables.

**Relationship to the manuscript.** Supplies every resource number, together with the assumptions that qualify it.

**Next.** `05_publication_exports.ipynb` regenerates the manuscript figures and tables from validated results only.